# SmolVLA 推論サーバ（REST API）

Colab の GPU 上で SmolVLA を動かし，ローカル PC の ROS 2 ノード（`vla_client`）から
**画像と関節角を送ると action chunk（50ステップ分の関節角）が返ってくる** API を立てます．

```
[ローカルPC: Gazebo + ROS 2]                         [Colab: GPU]
 vla_client ──(HTTPS: 画像, 関節角, タスク文)──▶ cloudflared ─▶ FastAPI + SmolVLA
            ◀────────(action chunk)──────────
```

- ランタイム → ランタイムのタイプを変更 → **GPU（T4 で可）**
- 上から順に実行し，最後に表示される `ros2 launch ...` のコマンドをローカル PC で実行します．

## 1. 環境構築（3〜5分）

In [2]:
!nvidia-smi

Sun Nov  1 19:44:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q "lerobot[smolvla,dataset,training]==0.6.1" fastapi uvicorn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. 使うモデル
講義では配布モデルを使います．自分で学習したモデルを使うときは `<ユーザー名>/smolvla_franka_push` に変更してください．
非公開のモデルを使う場合は，次のセルで Hugging Face にログインします．

In [4]:
MODEL_ID = "BugsB/smolvla_franka_push"  # ← 配布モデル名に変更

In [5]:
# 非公開のモデルを使うときは Hugging Face にログイン（左の🔑シークレットに HF_TOKEN を登録しておく）
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("ログインしました")
except Exception as e:
    print("ログインしませんでした（公開モデルならこのままでOK）:", e)

ログインしました


## 3. サーバのプログラム
中身を読んでみましょう．`/predict` で「前処理 → SmolVLA で推論 → 後処理（正規化を戻す）」をしています．

In [6]:
%%writefile smolvla_server.py
"""SmolVLA の推論用 REST API サーバ．

    python smolvla_server.py --model YOUR_HF_USERNAME/smolvla_franka_push --port 8000

エンドポイント
  GET  /health   : サーバとモデルの情報
  POST /predict  : 観測を送ると action chunk を返す
      入力 {"task": str, "state": [7], "images": {"front": base64 JPEG, "wrist": base64 JPEG}}
      出力 {"actions": [[7], ...], "inference_time": float}
"""
import argparse
import base64
import io
import time

import numpy as np
import torch
import uvicorn
from fastapi import FastAPI
from PIL import Image
from pydantic import BaseModel

from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

# ROS側のカメラ名 → データセット（モデル）上の画像キー
CAMERA_KEYS = {'front': 'observation.images.camera1', 'wrist': 'observation.images.camera2'}


class Observation(BaseModel):
    task: str
    state: list[float]
    images: dict[str, str]


def decode_image(b64: str) -> torch.Tensor:
    """base64 JPEG → (3, H, W) の float32 テンソル（0〜1）．"""
    img = Image.open(io.BytesIO(base64.b64decode(b64))).convert('RGB')
    return torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1).float() / 255.0


def create_app(model_id: str, device: str, n_steps: int | None = None) -> FastAPI:
    policy = SmolVLAPolicy.from_pretrained(model_id)
    policy.to(device)
    policy.eval()
    preprocess, postprocess = make_pre_post_processors(
        policy.config, pretrained_path=model_id,
        preprocessor_overrides={'device_processor': {'device': device}},
        postprocessor_overrides={'device_processor': {'device': 'cpu'}})
    chunk_size = policy.config.chunk_size
    app = FastAPI()

    @app.get('/health')
    def health():
        return {'model': model_id, 'device': device, 'chunk_size': chunk_size,
                'cameras': CAMERA_KEYS}

    @app.post('/predict')
    def predict(obs: Observation):
        t0 = time.time()
        batch = {'observation.state': torch.tensor(obs.state, dtype=torch.float32), 'task': obs.task}
        for name, key in CAMERA_KEYS.items():
            batch[key] = decode_image(obs.images[name])
        with torch.inference_mode():
            batch = preprocess(batch)
            policy.reset()  # 過去の観測・行動のキューを消す（毎回chunkを丸ごと推論する）
            chunk = policy.predict_action_chunk(batch)  # (1, chunk_size, 7) 正規化された値
            # 正規化を戻す（postprocess は1ステップ分 (1, 7) ずつ受け取る）
            actions = torch.stack([postprocess(chunk[:, i]) for i in range(chunk.shape[1])], dim=1)[0]
        if n_steps is not None:
            actions = actions[:n_steps]
        return {'actions': actions.cpu().numpy().tolist(), 'inference_time': time.time() - t0}

    return app


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model', required=True, help='HF Hub のモデルID またはローカルのパス')
    parser.add_argument('--device', default='cuda' if torch.cuda.is_available() else 'cpu')
    parser.add_argument('--host', default='0.0.0.0')
    parser.add_argument('--port', type=int, default=8000)
    args = parser.parse_args()
    uvicorn.run(create_app(args.model, args.device), host=args.host, port=args.port)


if __name__ == '__main__':
    main()

Writing smolvla_server.py


## 4. サーバの起動（モデルのダウンロードを含め1〜2分）

In [7]:
import subprocess, time, requests

server = subprocess.Popen(["python", "smolvla_server.py", "--model", MODEL_ID, "--port", "8000"],
                          stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        print(requests.get("http://localhost:8000/health", timeout=2).json())
        break
    except Exception:
        time.sleep(2)
else:
    print("起動に失敗しました．server.log を確認してください"); print(open("server.log").read()[-3000:])

{'model': 'BugsB/smolvla_franka_push', 'device': 'cuda', 'chunk_size': 50, 'cameras': {'front': 'observation.images.camera1', 'wrist': 'observation.images.camera2'}}


## 5. 動作確認（ダミー画像で推論）

In [8]:
import base64, io, numpy as np
from PIL import Image

def dummy_jpeg():
    buf = io.BytesIO(); Image.fromarray(np.zeros((256, 256, 3), np.uint8)).save(buf, format="JPEG")
    return base64.b64encode(buf.getvalue()).decode()

payload = {"task": "Push the red cube into the green goal area.",
           "state": [0.0, -0.3, 0.0, -2.2, 0.0, 1.9, 0.785],
           "images": {"front": dummy_jpeg(), "wrist": dummy_jpeg()}}
for _ in range(3):
    res = requests.post("http://localhost:8000/predict", json=payload).json()
    print(f"actions: {np.array(res['actions']).shape},  推論時間: {res['inference_time']:.3f}s")

actions: (50, 7),  推論時間: 2.165s
actions: (50, 7),  推論時間: 0.476s
actions: (50, 7),  推論時間: 0.494s


In [11]:
!pkill -f cloudflared

In [12]:
!netstat -tlpn | grep -E "8000|7860"

tcp        0      0 0.0.0.0:8000            0.0.0.0:*               LISTEN      2218/python3        


## 6. インターネットに公開（cloudflared）
Colab のサーバには外から直接アクセスできないので，[Cloudflare Tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/) で一時的な URL を発行します（アカウント不要）．
⚠️ URL を知っている人は誰でもこの API を使えます．講義が終わったら 8 でサーバを止めてください．

In [13]:
import re
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0); break
    time.sleep(1)
time.sleep(5)  # URL が使えるようになるまで少し待つ
print("サーバのURL:", url)
print(requests.get(f"{url}/health", timeout=30).json())
print("\n▼ ローカルPC（ROS 2 の環境）で実行するコマンド")
print(f"ros2 launch franka_vla_sim policy.launch.py server_url:={url}")

サーバのURL: https://frontier-enjoying-exercises-old.trycloudflare.com
{'model': 'BugsB/smolvla_franka_push', 'device': 'cuda', 'chunk_size': 50, 'cameras': {'front': 'observation.images.camera1', 'wrist': 'observation.images.camera2'}}

▼ ローカルPC（ROS 2 の環境）で実行するコマンド
ros2 launch franka_vla_sim policy.launch.py server_url:=https://frontier-enjoying-exercises-old.trycloudflare.com


## 7. ログの確認

In [10]:
print(open("server.log").read()[-3000:])

`torch_dtype` is deprecated! Use `dtype` instead!
Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...

Loading weights: 100%|██████████| 489/489 [00:00<00:00, 818.69it/s]
INFO:     Started server process [2218]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
Reducing the number of VLM layers to 16 ...
INFO:     127.0.0.1:43374 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:34084 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:34088 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:34092 - "POST /predict HTTP/1.1" 200 OK
INFO:     136.87.145.128:0 - "GET /health HTTP/1.1" 200 OK
INFO:     187.14.62.202:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     187.14.62.202:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     187.14.62.202:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     187.14.62.202:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     187.14.62.202:0 - "GET /

## 8. サーバの停止

In [14]:
tunnel.terminate(); server.terminate(); print("停止しました")

停止しました
